# Exploring the Online Retail II dataset

First look at the source data that will become the company's sales history in ERPNext.
Goal: understand its structure, coverage, and quality before designing the bootstrap.

## 1. Load the raw data

The workbook has one sheet per period (2009–2010 and 2010–2011).

In [6]:
from pathlib import Path

import pandas as pd

raw_file = Path("../data/raw/online_retail_II.xlsx")
sheets = pd.read_excel(raw_file, sheet_name=None)

for name, sheet in sheets.items():
    print(name, sheet.shape)

Year 2009-2010 (525461, 8)
Year 2010-2011 (541910, 8)


## 2. Combine the sheets and save as Parquet

Identifier columns are stored as text: stock codes and invoice numbers mix numbers and
letters (for example `85123A`, `C489449`), and Parquet requires one type per column.
Saved to `data/interim/` so later loads take seconds instead of minutes.

In [7]:
df = pd.concat(sheets.values(), ignore_index=True)

for col in ["Invoice", "StockCode", "Description", "Country"]:
    df[col] = df[col].astype("string")
df["Customer ID"] = df["Customer ID"].astype("Int64")

interim = Path("../data/interim")
interim.mkdir(parents=True, exist_ok=True)
df.to_parquet(interim / "online_retail.parquet")

df.shape

(1067371, 8)

## 3. First look

In [10]:
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom


In [27]:
df["Description"].value_counts().head(20)

Description
WHITE HANGING HEART T-LIGHT HOLDER    5918
REGENCY CAKESTAND 3 TIER              4412
JUMBO BAG RED RETROSPOT               3469
ASSORTED COLOUR BIRD ORNAMENT         2958
PARTY BUNTING                         2765
STRAWBERRY CERAMIC TRINKET BOX        2613
LUNCH BAG  BLACK SKULL.               2529
JUMBO STORAGE BAG SUKI                2434
HEART OF WICKER SMALL                 2319
JUMBO SHOPPER VINTAGE RED PAISLEY     2297
60 TEATIME FAIRY CAKE CASES           2271
PAPER CHAIN KIT 50'S CHRISTMAS        2215
LUNCH BAG SPACEBOY DESIGN             2206
REX CASH+CARRY JUMBO SHOPPER          2199
HOME BUILDING BLOCK WORD              2192
WOODEN FRAME ANTIQUE WHITE            2189
LUNCH BAG CARS BLUE                   2184
NATURAL SLATE HEART CHALKBOARD        2151
BAKING SET 9 PIECE RETROSPOT          2150
POSTAGE                               2115
Name: count, dtype: int64[pyarrow]

## 4. Coverage

In [14]:
df["InvoiceDate"].min(), df["InvoiceDate"].max()

(Timestamp('2009-12-01 07:45:00'), Timestamp('2011-12-09 12:50:00'))

The data covers 1 December 2009 to 9 December 2011, just over two years.

## 5. Data quality checks

In [19]:
df.isna().sum()

Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64

**Finding:** 243,007 rows (about 23%) have no Customer ID, and 4,382 rows have no description.

**Implication:** rows without a customer are real anonymous sales. They will be assigned to a
placeholder customer ("Guest Customer") so that revenue stays complete. Missing descriptions
still need checking against the unusual rows below.

In [20]:
cancelled = df["Invoice"].str.startswith("C")
cancelled.sum(), df.loc[cancelled, "Quantity"].describe()

(np.int64(19494),
 count    19494.000000
 mean       -25.186827
 std        805.104908
 min     -80995.000000
 25%         -6.000000
 50%         -2.000000
 75%         -1.000000
 max          1.000000
 Name: Quantity, dtype: float64)

**Finding:** 19,494 rows are cancellations (invoice numbers starting with "C"), almost all with
negative quantities. One cancellation of 80,995 units is a known data entry error. At least
one cancellation has a positive quantity, which is an anomaly.

**Implication:** cancellations become return invoices (credit notes) in ERPNext. The dataset
does not link a cancellation to its original invoice, so ERPNext's handling of standalone
returns must be checked.

In [21]:
(df["Quantity"] <= 0).sum(), (df["Price"] <= 0).sum()

(np.int64(22950), np.int64(6207))

**Finding:** 22,950 rows have a quantity of zero or less, but only 19,494 are cancellations,
so about 3,456 rows are other adjustments. 6,207 rows have a price of zero or less.

**Implication:** neither adjustments nor zero-price rows are sales, so both are excluded from
the sales history.

In [26]:
unusual = ~df["StockCode"].str.match(r"^\d{5}")
df.loc[unusual, ["StockCode", "Description"]].drop_duplicates().head(20)

,StockCode,Description
89,POST,POSTAGE
735,D,Discount
2377,DCGS0058,MISO PRETTY GUM
2378,DCGS0068,DOGS NIGHT COLLAR
2379,DOT,DOTCOM POSTAGE
2697,M,Manual
8371,DCGS0004,HAYNES CAMPER SHOULDER BAG
8373,DCGS0076,SUNJAR LED NIGHT NIGHT LIGHT
9292,C2,CARRIAGE
18410,BANK CHARGES,Bank Charges


**Finding:** codes that don't start with five digits fall into four groups: shipping charges
(POST, DOT, C2), discounts (D), internal entries (M, BANK CHARGES, TEST001), and real products
with unusual codes (DCGS items, gift vouchers).

**Implication:** a simple pattern can't separate products from non-products. The bootstrap
needs an explicit classification list of special codes.

## 6. Size and seasonality

In [23]:
df.duplicated().sum()

np.int64(34335)

In [24]:
df["Customer ID"].nunique(), df["StockCode"].nunique(), df["Country"].nunique()

(5942, 5305, 43)

In [25]:
monthly = df.groupby(df["InvoiceDate"].dt.to_period("M"))["Invoice"].nunique()
monthly

InvoiceDate
2009-12    2330
2010-01    1633
2010-02    1969
2010-03    2367
2010-04    1892
2010-05    2418
2010-06    2216
2010-07    2017
2010-08    1877
2010-09    2375
2010-10    2965
2010-11    3669
2010-12    2025
2011-01    1476
2011-02    1393
2011-03    1983
2011-04    1744
2011-05    2162
2011-06    2012
2011-07    1927
2011-08    1737
2011-09    2327
2011-10    2637
2011-11    3462
2011-12    1015
Freq: M, Name: Invoice, dtype: int64

## 7. Findings

| Finding | Handling in the bootstrap |
|---|---|
| 23% of rows have no customer | Assign to a placeholder "Guest Customer" |
| 19,494 cancellations | Load as return invoices (credit notes) |
| About 3,456 non-cancellation adjustments | Exclude |
| 6,207 rows with zero or negative price | Exclude |
| Shipping charges (POST, DOT, C2) | Load as a non-stock service item |
| Discounts (D) | To decide |
| Internal entries (M, BANK CHARGES, TEST001, TEST002) | Exclude |
| Real products with unusual codes (DCGS, gift vouchers) | Keep |
| One cancellation of 80,995 units | Known error; flag as an outlier |
| 34,335 exact duplicate rows | Remove before loading |
| 5,942 customers, 5,305 products, 43 countries | Load all as master data |
| Strong seasonality, peaking in November | Load November 2011 as the one-month window |
